# M3 · Copy-time calibration and dense-model baselines

**Goal 1:** measure how long it takes to copy one expert (12.6 MB) from CPU to GPU.
The simulator uses this number to turn "misses" into "seconds".

**Goal 2:** compare OLMoE with ordinary *dense* models (no experts) of the same family:

| Model | Total size | Used per token |
|---|---|---|
| OLMo-1B (dense) | 1.2B | 1.2B |
| **OLMoE-1B-7B** | 6.9B | 1.3B |
| OLMo-7B (dense, "vanilla") | 6.9B | 6.9B |

**Kaggle:** GPU T4 x1, Internet On. About 1.5–2 hours (3 model downloads).
Run the cells one model at a time; results are saved after each model.

In [ ]:
# ---- Setup: get the latest project code (run this cell first, every session) ----
import os, sys

REPO_URL = "https://github.com/YOUR-TEAM/moe-offload.git"   # <-- change to your team's repo
# Private repo? Add a Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets), then uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")

if os.path.exists("/kaggle"):                      # running on Kaggle
    import glob, shutil
    CODE_DIR = "/kaggle/working/moe-offload"
    # Option A: the code was uploaded as a Kaggle dataset ("moe-offload-code") and added as an input
    uploaded = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/moe-offload-code/config.py", recursive=True)]
    if uploaded:
        shutil.copytree(uploaded[0], CODE_DIR, dirs_exist_ok=True)
        print("using code from dataset:", uploaded[0])
    # Option B: get the code from GitHub
    elif not os.path.exists(CODE_DIR):
        !git clone -q {REPO_URL} {CODE_DIR}         # first time: download the code
    else:
        !git -C {CODE_DIR} pull -q                  # later: get the newest commits
    os.chdir(CODE_DIR)
    !pip install -q -r requirements.txt
else:                                              # running on your own computer
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                             # go to the repo root

sys.path.insert(0, os.getcwd())                    # so "import config" and "from src..." work
!git log -1 --format="Code version: %h  %s"

## 1. Copy time for one expert
*Pinned* memory is RAM that the GPU can read directly, so copies are faster.

In [ ]:
import json
from src.calibrate import calibrate

calib = calibrate()            # also saves results/calibration.json
print(json.dumps(calib, indent=2))
print(f"\n=> copying one expert takes {calib['copy_time_s']*1000:.2f} ms")
print(f"=> one token needs 16 layers x 8 experts = 128 experts; with NO cache that is "
      f"{128*calib['copy_time_s']*1000:.0f} ms per token just for copying")

## 2. Baselines
For each model `run_baseline` measures size, FLOPs per token, perplexity on both
datasets, speed (tokens/second), GPU memory and energy (joules per token).
Each model is loaded, measured, and removed from the GPU before the next one.

In [ ]:
import os
import pandas as pd
import config
from src.baselines import run_baseline

csv_path = os.path.join(config.RESULTS_DIR, "baselines.csv")
rows = []

def measure(model_id):
    """Measure one model and save the table so far (in case the session stops)."""
    rows.append(run_baseline(model_id))
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    return pd.DataFrame(rows).T

In [ ]:
measure(config.DENSE_SMALL)     # OLMo-1B dense, ~5 min

In [ ]:
measure(config.MOE_MODEL)       # OLMoE-1B-7B, ~15 min

The dense 7B model is ~13.8 GB in float16, which only just fits on the T4.
If it runs out of memory, `run_baseline` automatically loads it in **8-bit**
(the `precision` column says which). Mention this in the report.

In [ ]:
# 8-bit fallback needs bitsandbytes; install it only for this step
!pip install -q bitsandbytes
measure(config.DENSE_LARGE)     # OLMo-7B dense, ~20 min

## 3. The gain table: OLMoE vs dense 7B

In [ ]:
df = pd.read_csv(csv_path)
df["name"] = df["model"].str.split("/").str[-1]
df = df.set_index("name")

cols = ["precision", "total_params_B", "active_params_B", "GFLOPs_per_token",
        "wikitext_perplexity", "code_perplexity", "weights_GB", "peak_GPU_GB",
        "decode_tok_per_s", "decode_J_per_tok", "wikitext_prefill_J_per_tok"]
display(df[cols].round(3).T)

moe, dense = df.loc["OLMoE-1B-7B-0924"], df.loc["OLMo-7B-0724-hf"]
print("Compute per token : OLMoE uses %.1fx fewer FLOPs than dense 7B" % (dense.GFLOPs_per_token / moe.GFLOPs_per_token))
print("Prefill energy    : OLMoE uses %.1fx less energy per token" % (dense.wikitext_prefill_J_per_tok / moe.wikitext_prefill_J_per_tok))
print("Prefill speed     : OLMoE is %.1fx faster" % (moe.wikitext_prefill_tok_per_s / dense.wikitext_prefill_tok_per_s))

## 4. Bar charts
Green = OLMoE, grey = dense models.

In [ ]:
import matplotlib.pyplot as plt

show = ["GFLOPs_per_token", "wikitext_perplexity", "code_perplexity",
        "peak_GPU_GB", "decode_tok_per_s", "decode_J_per_tok"]
fig, axes = plt.subplots(1, len(show), figsize=(20, 3.5))
colors = ["tab:green" if "OLMoE" in n else "tab:gray" for n in df.index]
for ax, col in zip(axes, show):
    ax.bar(df.index, df[col], color=colors)
    ax.set_title(col, fontsize=10)
    ax.tick_params(axis="x", rotation=60, labelsize=8)
plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, "M3_baselines.png"), dpi=130)
plt.show()

## What to write in `docs/M3.md`
- Copy time per expert and bandwidth (GB/s), pinned vs normal memory.
- OLMoE vs dense 7B: compute, energy, speed, quality.
- **Note:** the full OLMoE still needs ~14 GB of GPU memory. M8 shows how
  offloading + a cache brings that down to ~4 GB.

## ✅ Done when
`results/calibration.json` and `results/baselines.csv` exist (save the notebook
version so later notebooks can add this output as an input).